# ML3 — E-commerce Product Recommendation

So sánh ba phương pháp gợi ý sản phẩm trên Retailrocket với sự kiện `view`:

1. **Popularity** — xếp hạng theo số lượt xem trong tập train.
2. **Item-CF** — cosine similarity trên ma trận tương tác user–item nhị phân.
3. **SASRec** — mô hình gợi ý tuần tự dùng causal self-attention.

## Quy trình tái lập

Chạy tuần tự các cell từ trên xuống dưới. Pipeline này được rút gọn từ notebook làm việc, loại các cell chẩn đoán/cell cũ bị trùng và giữ pipeline `shared` đã dùng để tạo kết quả báo cáo.

**Lưu ý:** Dữ liệu và checkpoint không được lưu trong repository. Notebook làm việc với Google Drive tại `/content/drive/MyDrive/BigData_ML3_ToNhu`. Đặt `clean_events_final.csv` vào thư mục `raw_data/` trước khi chạy. Các file trung gian và checkpoint sẽ được tạo trong Drive.


## 0. Môi trường và thư mục dự án

Khuyến nghị mở bằng Google Colab và chọn GPU (T4/L4 nếu khả dụng). Colab thường đã cài NumPy, pandas và PyTorch; cần `pyarrow` để đọc/ghi Parquet và `scipy` để xử lý ma trận sparse.

Nếu thiếu thư viện, chạy trong cell riêng: `!pip -q install pyarrow scipy`.


In [2]:
from pathlib import Path
import os, sys, platform

# Kết nối Google Drive theo cách an toàn khi Drive đã được mount.
MOUNT_DIR = Path("/content/drive")
MYDRIVE_DIR = MOUNT_DIR / "MyDrive"

if MYDRIVE_DIR.is_dir():
    print("Google Drive đã khả dụng; bỏ qua lệnh mount.")
else:
    from google.colab import drive

    # Không xóa nội dung ở mountpoint. Nếu thư mục đã có nội dung
    # nhưng chưa có MyDrive, dừng và yêu cầu làm mới runtime an toàn.
    if MOUNT_DIR.exists() and any(MOUNT_DIR.iterdir()):
        raise RuntimeError(
            "/content/drive đã tồn tại và không rỗng nhưng chưa thấy MyDrive. "
            "Không xóa thư mục này. Vào Runtime > Disconnect and delete runtime, "
            "kết nối lại rồi chạy notebook từ đầu."
        )

    drive.mount("/content/drive")

if not MYDRIVE_DIR.is_dir():
    raise FileNotFoundError(
        "Không thấy /content/drive/MyDrive sau khi kết nối Google Drive. "
        "Kiểm tra quyền truy cập tài khoản Drive rồi chạy lại cell này."
    )

import numpy as np
import pandas as pd
import torch
import scipy
import pyarrow

BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
RAW_DIR = os.path.join(BASE_DIR, "raw_data")
DATA_DIR = os.path.join(BASE_DIR, "data_checkpoint")
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")
ITEMCF_DIR = os.path.join(BASE_DIR, "itemcf_shared")
SASREC_DIR = os.path.join(BASE_DIR, "sasrec_shared")
RESULTS_DIR = os.path.join(BASE_DIR, "results_shared")

for folder in [RAW_DIR, DATA_DIR, EVAL_DIR, ITEMCF_DIR, SASREC_DIR, RESULTS_DIR]:
    os.makedirs(folder, exist_ok=True)

clean_file = os.path.join(RAW_DIR, "clean_events_final.csv")
if not os.path.isfile(clean_file):
    raise FileNotFoundError(
        f"Không tìm thấy dữ liệu sạch: {clean_file}. "
        "Hãy đặt clean_events_final.csv vào thư mục raw_data trong Drive."
    )

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("PyArrow:", pyarrow.__version__)
print("PyTorch:", torch.__version__)
print("GPU khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Dữ liệu sạch:", clean_file)
print("BASE_DIR:", BASE_DIR)


Mounted at /content/drive
Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
NumPy: 2.1.3
pandas: 2.2.3
SciPy: 1.16.3
PyArrow: 23.0.1
PyTorch: 2.11.0+cu130
GPU khả dụng: True
GPU: NVIDIA L4
Dữ liệu sạch: /content/drive/MyDrive/BigData_ML3_ToNhu/raw_data/clean_events_final.csv
BASE_DIR: /content/drive/MyDrive/BigData_ML3_ToNhu


## 1. Lọc view và chia train / validation / test

- Chỉ giữ sự kiện `view`.
- Giữ khách hàng có ít nhất 3 lượt xem.
- Sắp xếp theo `visitorid`, `timestamp` bằng stable sort (`mergesort`). Nếu timestamp trùng nhau, thứ tự hàng gốc được giữ lại; dữ liệu không có trường thứ tự sự kiện bổ sung để phân giải ngữ nghĩa của các timestamp trùng.
- Hai tương tác cuối của mỗi khách hàng làm validation/test target; các tương tác trước đó làm train.


In [3]:
import os
import pandas as pd

# 1. Đường dẫn dữ liệu chung và checkpoint mới
clean_file = (
    "/content/drive/MyDrive/BigData_ML3_ToNhu/"
    "raw_data/clean_events_final.csv"
)

checkpoint_dir = (
    "/content/drive/MyDrive/BigData_ML3_ToNhu/"
    "data_checkpoint"
)

os.makedirs(checkpoint_dir, exist_ok=True)

shared_view_file = os.path.join(
    checkpoint_dir,
    "view_events_eligible_shared.parquet"
)

# 2. Đọc dữ liệu theo từng phần, chỉ giữ hành vi view
view_parts = []

for chunk in pd.read_csv(
    clean_file,
    usecols=[
        "timestamp",
        "datetime",
        "visitorid",
        "session_id",
        "event",
        "itemid"
    ],
    chunksize=300_000
):
    view_chunk = chunk.loc[
        chunk["event"].eq("view")
    ].copy()

    view_parts.append(view_chunk)

views = pd.concat(view_parts, ignore_index=True)

print("Số lượt view trước khi lọc khách hàng:", len(views))
print("Số khách hàng có lượt view:", views["visitorid"].nunique())
print("Số sản phẩm được xem:", views["itemid"].nunique())

# 3. Kiểm tra dữ liệu thiếu trước khi tiếp tục
required_cols = ["timestamp", "visitorid", "itemid"]
missing_counts = views[required_cols].isna().sum()

print("\nGiá trị thiếu ở các cột bắt buộc:")
print(missing_counts)

if missing_counts.any():
    raise ValueError(
        "Có giá trị thiếu ở cột bắt buộc. "
        "Dừng để kiểm tra, chưa lưu checkpoint."
    )

# 4. Sắp xếp tương tác theo khách hàng và thời gian
views = views.sort_values(
    ["visitorid", "timestamp"],
    kind="mergesort"
).reset_index(drop=True)

# 5. Chỉ giữ khách hàng có ít nhất 3 lượt view
view_counts = views.groupby("visitorid").size()
eligible_users = view_counts[view_counts >= 3].index

views_eligible_shared = views[
    views["visitorid"].isin(eligible_users)
].copy().reset_index(drop=True)

# 6. Lưu thành file mới, không ghi đè checkpoint cũ
views_eligible_shared.to_parquet(
    shared_view_file,
    index=False
)

print("\n===== KẾT QUẢ CHECKPOINT MỚI =====")
print("Số tương tác:", len(views_eligible_shared))
print("Số khách hàng:", views_eligible_shared["visitorid"].nunique())
print("Số sản phẩm:", views_eligible_shared["itemid"].nunique())
print("Đường dẫn lưu:", shared_view_file)
print(
    "Kích thước file (MB):",
    round(os.path.getsize(shared_view_file) / (1024 ** 2), 2)
)

Số lượt view trước khi lọc khách hàng: 2664215
Số khách hàng có lượt view: 1404178
Số sản phẩm được xem: 234838

Giá trị thiếu ở các cột bắt buộc:
timestamp    0
visitorid    0
itemid       0
dtype: int64

===== KẾT QUẢ CHECKPOINT MỚI =====
Số tương tác: 1245580
Số khách hàng: 190980
Số sản phẩm: 133474
Đường dẫn lưu: /content/drive/MyDrive/BigData_ML3_ToNhu/data_checkpoint/view_events_eligible_shared.parquet
Kích thước file (MB): 29.39


In [4]:
import os
import pandas as pd

# Đường dẫn checkpoint mới từ dữ liệu chung
BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
DATA_DIR = os.path.join(BASE_DIR, "data_checkpoint")

shared_file = os.path.join(
    DATA_DIR,
    "view_events_eligible_shared.parquet"
)

# Đọc dữ liệu đã lọc
views = pd.read_parquet(shared_file)

# Sắp xếp theo khách hàng và thời gian
views = views.sort_values(
    ["visitorid", "timestamp"],
    kind="mergesort"
).reset_index(drop=True)

# Xác định vị trí từng tương tác trong chuỗi của mỗi khách hàng
grouped = views.groupby("visitorid", sort=False)
position = grouped.cumcount()
sequence_length = grouped["itemid"].transform("size")

# Chia train / validation / test
train_df = views[position < sequence_length - 2].copy()
valid_df = views[position == sequence_length - 2].copy()
test_df = views[position == sequence_length - 1].copy()

# Chỉ lưu các cột cần thiết cho bước Recommendation tiếp theo
train_df = train_df[["visitorid", "itemid", "timestamp"]]
valid_df = valid_df[["visitorid", "itemid", "timestamp"]]
test_df = test_df[["visitorid", "itemid", "timestamp"]]

# Lưu thành file mới, không ghi đè file cũ
train_file = os.path.join(DATA_DIR, "train_interactions_shared.parquet")
valid_file = os.path.join(DATA_DIR, "valid_targets_shared.parquet")
test_file = os.path.join(DATA_DIR, "test_targets_shared.parquet")

train_df.to_parquet(train_file, index=False)
valid_df.to_parquet(valid_file, index=False)
test_df.to_parquet(test_file, index=False)

# Kiểm tra độ phủ sản phẩm trong tập train
train_catalog = set(train_df["itemid"].unique())

valid_covered = valid_df["itemid"].isin(train_catalog)
test_covered = test_df["itemid"].isin(train_catalog)

print("===== KẾT QUẢ CHIA DỮ LIỆU CHUNG =====")
print("Số khách hàng:", views["visitorid"].nunique())
print("Số tương tác train:", len(train_df))
print("Số khách hàng validation:", len(valid_df))
print("Số khách hàng test:", len(test_df))
print("Số sản phẩm trong train catalog:", len(train_catalog))

print("\n===== ĐỘ PHỦ SẢN PHẨM =====")
print(
    "Validation target thuộc train catalog:",
    f"{valid_covered.sum():,}/{len(valid_df):,}",
    f"({valid_covered.mean() * 100:.4f}%)"
)
print(
    "Test target thuộc train catalog:",
    f"{test_covered.sum():,}/{len(test_df):,}",
    f"({test_covered.mean() * 100:.4f}%)"
)

print("\n===== FILE ĐÃ LƯU =====")
print(train_file)
print(valid_file)
print(test_file)

===== KẾT QUẢ CHIA DỮ LIỆU CHUNG =====
Số khách hàng: 190980
Số tương tác train: 863620
Số khách hàng validation: 190980
Số khách hàng test: 190980
Số sản phẩm trong train catalog: 114800

===== ĐỘ PHỦ SẢN PHẨM =====
Validation target thuộc train catalog: 180,572/190,980 (94.5502%)
Test target thuộc train catalog: 178,577/190,980 (93.5056%)

===== FILE ĐÃ LƯU =====
/content/drive/MyDrive/BigData_ML3_ToNhu/data_checkpoint/train_interactions_shared.parquet
/content/drive/MyDrive/BigData_ML3_ToNhu/data_checkpoint/valid_targets_shared.parquet
/content/drive/MyDrive/BigData_ML3_ToNhu/data_checkpoint/test_targets_shared.parquet


## 2. Tạo bộ ứng viên cố định

Mỗi user có một positive target ở cột 0 và 100 negative candidates. Chỉ đánh giá các user có cả target validation và test nằm trong train catalog. Seed lấy mẫu negative: `2026`.


In [5]:
import os
import numpy as np
import pandas as pd

# 1. Đường dẫn dữ liệu chung
BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
DATA_DIR = os.path.join(BASE_DIR, "data_checkpoint")
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")
os.makedirs(EVAL_DIR, exist_ok=True)

# 2. Đọc ba tập đã tạo từ dữ liệu chung
train_df = pd.read_parquet(
    os.path.join(DATA_DIR, "train_interactions_shared.parquet")
)
valid_df = pd.read_parquet(
    os.path.join(DATA_DIR, "valid_targets_shared.parquet")
)
test_df = pd.read_parquet(
    os.path.join(DATA_DIR, "test_targets_shared.parquet")
)

# 3. Xác định danh mục sản phẩm trong train
catalog = np.sort(train_df["itemid"].unique()).astype(np.int32)
catalog_set = set(catalog.tolist())

valid_map = dict(zip(
    valid_df["visitorid"].astype(int),
    valid_df["itemid"].astype(int)
))
test_map = dict(zip(
    test_df["visitorid"].astype(int),
    test_df["itemid"].astype(int)
))

# Chỉ giữ user có cả hai mục tiêu trong train catalog
eval_users = np.array([
    uid for uid in sorted(valid_map.keys() & test_map.keys())
    if valid_map[uid] in catalog_set
    and test_map[uid] in catalog_set
], dtype=np.int64)

print("Số khách hàng đủ điều kiện đánh giá:", len(eval_users))
print("Kích thước train catalog:", len(catalog))

# 4. Lập lịch sử sản phẩm đã xem trong train của từng user
train_history = {}

for uid, item in zip(
    train_df["visitorid"].to_numpy(),
    train_df["itemid"].to_numpy()
):
    train_history.setdefault(int(uid), set()).add(int(item))

# 5. Hàm lấy 100 negative items không trùng nhau
rng = np.random.default_rng(2026)
N_NEGATIVES = 100
N_CANDIDATES = 1 + N_NEGATIVES

def sample_negatives(forbidden):
    negatives = []
    selected = set()

    while len(negatives) < N_NEGATIVES:
        remaining = N_NEGATIVES - len(negatives)
        batch_size = max(128, remaining * 2)

        draws = catalog[
            rng.integers(0, len(catalog), size=batch_size)
        ]

        for item in draws:
            item = int(item)

            if item in forbidden or item in selected:
                continue

            selected.add(item)
            negatives.append(item)

            if len(negatives) == N_NEGATIVES:
                break

    return negatives

# 6. Tạo hai bộ ứng viên
valid_candidates = np.empty(
    (len(eval_users), N_CANDIDATES), dtype=np.int32
)
test_candidates = np.empty(
    (len(eval_users), N_CANDIDATES), dtype=np.int32
)

for idx, uid in enumerate(eval_users):
    valid_item = valid_map[int(uid)]
    test_item = test_map[int(uid)]

    history = train_history.get(int(uid), set())

    # Validation: negative không nằm trong train history
    valid_forbidden = set(history)
    valid_forbidden.add(valid_item)

    valid_candidates[idx, 0] = valid_item
    valid_candidates[idx, 1:] = sample_negatives(valid_forbidden)

    # Test: negative không nằm trong train hoặc validation history
    test_forbidden = set(history)
    test_forbidden.add(valid_item)
    test_forbidden.add(test_item)

    test_candidates[idx, 0] = test_item
    test_candidates[idx, 1:] = sample_negatives(test_forbidden)

    if (idx + 1) % 25000 == 0:
        print(f"Đã tạo ứng viên cho {idx + 1:,} khách hàng")

# 7. Kiểm tra kết quả trước khi lưu
assert valid_candidates.shape == (len(eval_users), 101)
assert test_candidates.shape == (len(eval_users), 101)

assert np.array_equal(
    valid_candidates[:, 0],
    np.array([valid_map[int(u)] for u in eval_users], dtype=np.int32)
)
assert np.array_equal(
    test_candidates[:, 0],
    np.array([test_map[int(u)] for u in eval_users], dtype=np.int32)
)

# 8. Lưu thành file mới, không ghi đè kết quả cũ
np.save(
    os.path.join(EVAL_DIR, "eval_users_shared.npy"),
    eval_users
)
np.save(
    os.path.join(EVAL_DIR, "valid_candidates_shared.npy"),
    valid_candidates
)
np.save(
    os.path.join(EVAL_DIR, "test_candidates_shared.npy"),
    test_candidates
)

print("\n===== KẾT QUẢ BỘ ỨNG VIÊN MỚI =====")
print("Số khách hàng:", len(eval_users))
print("Validation candidates:", valid_candidates.shape)
print("Test candidates:", test_candidates.shape)
print("Kiểu dữ liệu:", valid_candidates.dtype)
print("Sản phẩm mục tiêu nằm ở cột đầu:", True)
print("Seed lấy mẫu:", 2026)
print("\nĐã lưu tại:", EVAL_DIR)

Số khách hàng đủ điều kiện đánh giá: 172513
Kích thước train catalog: 114800
Đã tạo ứng viên cho 25,000 khách hàng
Đã tạo ứng viên cho 50,000 khách hàng
Đã tạo ứng viên cho 75,000 khách hàng
Đã tạo ứng viên cho 100,000 khách hàng
Đã tạo ứng viên cho 125,000 khách hàng
Đã tạo ứng viên cho 150,000 khách hàng

===== KẾT QUẢ BỘ ỨNG VIÊN MỚI =====
Số khách hàng: 172513
Validation candidates: (172513, 101)
Test candidates: (172513, 101)
Kiểu dữ liệu: int32
Sản phẩm mục tiêu nằm ở cột đầu: True
Seed lấy mẫu: 2026

Đã lưu tại: /content/drive/MyDrive/BigData_ML3_ToNhu/evaluation


## 3. Phá hòa dùng chung

Khi hai ứng viên có cùng điểm, dùng các ma trận ưu tiên ngẫu nhiên được tạo cố định để bảo đảm cách xử lý hòa điểm nhất quán giữa các mô hình.

- Validation priority seed: `20261010`
- Test priority seed: `20261011`


In [6]:
import os
import numpy as np

BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")

eval_users = np.load(
    os.path.join(EVAL_DIR, "eval_users_shared.npy")
)

N_CANDIDATES = 101

def create_tie_priorities(n_users, seed):
    rng = np.random.default_rng(seed)
    priorities = np.empty(
        (n_users, N_CANDIDATES),
        dtype=np.int16
    )

    for i in range(n_users):
        # Mỗi hàng là một hoán vị duy nhất từ 0 đến 100.
        # Giá trị nhỏ hơn sẽ được ưu tiên khi điểm bằng nhau.
        priorities[i] = rng.permutation(N_CANDIDATES)

    return priorities

valid_candidates = np.load(
    os.path.join(EVAL_DIR, "valid_candidates_shared.npy"),
    mmap_mode="r"
)
test_candidates = np.load(
    os.path.join(EVAL_DIR, "test_candidates_shared.npy"),
    mmap_mode="r"
)

assert valid_candidates.shape == (len(eval_users), 101)
assert test_candidates.shape == (len(eval_users), 101)

valid_priorities = create_tie_priorities(
    len(eval_users), seed=20261010
)
test_priorities = create_tie_priorities(
    len(eval_users), seed=20261011
)

np.save(
    os.path.join(EVAL_DIR, "tie_priorities_valid_shared.npy"),
    valid_priorities
)
np.save(
    os.path.join(EVAL_DIR, "tie_priorities_test_shared.npy"),
    test_priorities
)

assert np.all(
    np.sort(test_priorities, axis=1)
    == np.arange(101, dtype=np.int16)
)

print("Số khách hàng:", len(eval_users))
print("Validation priorities:", valid_priorities.shape)
print("Test priorities:", test_priorities.shape)
print("Mỗi hàng có 101 thứ tự ưu tiên khác nhau.")
print("Đã lưu bộ phá hòa dùng chung tại:", EVAL_DIR)

Số khách hàng: 172513
Validation priorities: (172513, 101)
Test priorities: (172513, 101)
Mỗi hàng có 101 thứ tự ưu tiên khác nhau.
Đã lưu bộ phá hòa dùng chung tại: /content/drive/MyDrive/BigData_ML3_ToNhu/evaluation


## 4. Popularity baseline

Popularity đếm lượt `view` trong tập train; lặp lại lượt xem của cùng item vẫn được tính vào tần suất sự kiện. Đánh giá validation và test với tie priorities dùng chung.


In [7]:
import os
import json
import numpy as np
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
DATA_DIR = os.path.join(BASE_DIR, "data_checkpoint")
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")
RESULTS_DIR = os.path.join(BASE_DIR, "results_shared")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Đọc train và bộ ứng viên cố định
train_df = pd.read_parquet(
    os.path.join(DATA_DIR, "train_interactions_shared.parquet")
)

valid_candidates = np.load(
    os.path.join(EVAL_DIR, "valid_candidates_shared.npy"),
    mmap_mode="r"
)
test_candidates = np.load(
    os.path.join(EVAL_DIR, "test_candidates_shared.npy"),
    mmap_mode="r"
)

valid_priorities = np.load(
    os.path.join(EVAL_DIR, "tie_priorities_valid_shared.npy"),
    mmap_mode="r"
)
test_priorities = np.load(
    os.path.join(EVAL_DIR, "tie_priorities_test_shared.npy"),
    mmap_mode="r"
)

assert valid_candidates.shape == valid_priorities.shape
assert test_candidates.shape == test_priorities.shape
assert valid_candidates.shape[1] == 101
assert test_candidates.shape[1] == 101

# Độ phổ biến được tính bằng số lượt view trong train
popularity = train_df["itemid"].value_counts().to_dict()

def evaluate_popularity_fair(candidates, priorities, split_name):
    n_users = len(candidates)
    ranks = np.empty(n_users, dtype=np.int16)
    batch_size = 5000

    for start in range(0, n_users, batch_size):
        end = min(start + batch_size, n_users)

        cand_batch = np.asarray(candidates[start:end])
        priority_batch = np.asarray(priorities[start:end])

        # Lấy điểm popularity cho từng ứng viên
        scores = np.fromiter(
            (
                popularity.get(int(item), 0)
                for item in cand_batch.ravel()
            ),
            dtype=np.int32,
            count=cand_batch.size
        ).reshape(cand_batch.shape)

        positive_score = scores[:, [0]]
        positive_priority = priority_batch[:, [0]]

        # Ứng viên đứng trước target nếu:
        # 1) điểm cao hơn; hoặc
        # 2) điểm bằng nhau và priority nhỏ hơn target.
        higher_score = scores[:, 1:] > positive_score
        tied_but_preferred = (
            (scores[:, 1:] == positive_score)
            & (priority_batch[:, 1:] < positive_priority)
        )

        ranks[start:end] = (
            1 + np.sum(
                higher_score | tied_but_preferred,
                axis=1
            )
        )

        if end % 25000 < batch_size or end == n_users:
            print(
                f"{split_name}: đã đánh giá "
                f"{end:,}/{n_users:,} khách hàng"
            )

    hits = ranks <= 10
    ndcg = np.where(
        hits,
        1.0 / np.log2(ranks + 1),
        0.0
    )

    metrics = {
        "model": "Popularity",
        "split": split_name,
        "tie_breaking": "shared_random_priorities",
        "num_users": int(n_users),
        "num_candidates_per_user": 101,
        "HR@10": float(hits.mean()),
        "NDCG@10": float(ndcg.mean()),
        "rank_1": int(np.sum(ranks == 1)),
        "rank_le_10": int(np.sum(ranks <= 10)),
        "misses": int(np.sum(ranks > 10)),
        "median_rank": float(np.median(ranks)),
        "mean_rank": float(np.mean(ranks)),
        "max_rank": int(np.max(ranks))
    }

    suffix = split_name.lower()

    np.save(
        os.path.join(
            RESULTS_DIR,
            f"popularity_{suffix}_positive_rank_shared_fair.npy"
        ),
        ranks
    )

    with open(
        os.path.join(
            RESULTS_DIR,
            f"popularity_{suffix}_metrics_shared_fair.json"
        ),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    return metrics

# Tính lại validation và test bằng cùng quy tắc phá hòa
valid_metrics_fair = evaluate_popularity_fair(
    valid_candidates, valid_priorities, "VALID"
)

test_metrics_fair = evaluate_popularity_fair(
    test_candidates, test_priorities, "TEST"
)

print("\n===== POPULARITY — VALIDATION, PHÁ HÒA CHUNG =====")
print(json.dumps(valid_metrics_fair, ensure_ascii=False, indent=2))

print("\n===== POPULARITY — TEST, PHÁ HÒA CHUNG =====")
print(json.dumps(test_metrics_fair, ensure_ascii=False, indent=2))

print("\nKết quả mới được lưu tại:", RESULTS_DIR)

VALID: đã đánh giá 25,000/172,513 khách hàng
VALID: đã đánh giá 50,000/172,513 khách hàng
VALID: đã đánh giá 75,000/172,513 khách hàng
VALID: đã đánh giá 100,000/172,513 khách hàng
VALID: đã đánh giá 125,000/172,513 khách hàng
VALID: đã đánh giá 150,000/172,513 khách hàng
VALID: đã đánh giá 172,513/172,513 khách hàng
TEST: đã đánh giá 25,000/172,513 khách hàng
TEST: đã đánh giá 50,000/172,513 khách hàng
TEST: đã đánh giá 75,000/172,513 khách hàng
TEST: đã đánh giá 100,000/172,513 khách hàng
TEST: đã đánh giá 125,000/172,513 khách hàng
TEST: đã đánh giá 150,000/172,513 khách hàng
TEST: đã đánh giá 172,513/172,513 khách hàng

===== POPULARITY — VALIDATION, PHÁ HÒA CHUNG =====
{
  "model": "Popularity",
  "split": "VALID",
  "tie_breaking": "shared_random_priorities",
  "num_users": 172513,
  "num_candidates_per_user": 101,
  "HR@10": 0.5480804345179784,
  "NDCG@10": 0.3253415822982788,
  "rank_1": 25978,
  "rank_le_10": 94551,
  "misses": 77962,
  "median_rank": 9.0,
  "mean_rank": 17.60

## 5. Item-based Collaborative Filtering (Item-CF)

Item-CF khử trùng cặp user–item thành ma trận nhị phân, tính co-occurrence và cosine similarity. Baseline đánh giá test dùng cùng candidate set và test tie priorities.


In [8]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

# 1. Đường dẫn
BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
DATA_DIR = os.path.join(BASE_DIR, "data_checkpoint")
ITEMCF_DIR = os.path.join(BASE_DIR, "itemcf_shared")
os.makedirs(ITEMCF_DIR, exist_ok=True)

# 2. Đọc dữ liệu train từ nguồn chung
train_df = pd.read_parquet(
    os.path.join(DATA_DIR, "train_interactions_shared.parquet")
)

# 3. Tạo danh mục và danh sách khách hàng theo thứ tự cố định
catalog = np.sort(train_df["itemid"].unique()).astype(np.int32)
user_ids = np.sort(train_df["visitorid"].unique()).astype(np.int64)

# Item-CF dùng ma trận nhị phân: một khách hàng có xem một sản phẩm
# hay không; nhiều lượt xem cùng sản phẩm chỉ tính một lần.
train_pairs = train_df[["visitorid", "itemid"]].drop_duplicates()

rows = np.searchsorted(
    user_ids, train_pairs["visitorid"].to_numpy()
)
cols = np.searchsorted(
    catalog, train_pairs["itemid"].to_numpy()
)

X = csr_matrix(
    (
        np.ones(len(train_pairs), dtype=np.int32),
        (rows, cols)
    ),
    shape=(len(user_ids), len(catalog)),
    dtype=np.int32
)
X.sum_duplicates()
X.data[:] = 1
X.eliminate_zeros()

print("===== MA TRẬN KHÁCH HÀNG - SẢN PHẨM =====")
print("Kích thước:", X.shape)
print("Số cặp user-item khác nhau:", X.nnz)

# 4. Tính đồng xuất hiện sản phẩm: C = X.T @ X
print("\nĐang tính ma trận đồng xuất hiện...")
C = (X.T @ X).tocsr()
C.setdiag(0)
C.eliminate_zeros()

print("Kích thước co-occurrence:", C.shape)
print("Số phần tử khác 0:", C.nnz)

# 5. Tính cosine similarity từ co-occurrence
print("\nĐang tính cosine similarity...")
item_counts = np.asarray(X.sum(axis=0)).ravel().astype(np.float64)
norms = np.sqrt(item_counts)
norms[norms == 0] = 1.0

inv_norms = (1.0 / norms).astype(np.float32)

S = C.astype(np.float32)
S = S.multiply(inv_norms[:, None])
S = S.multiply(inv_norms[None, :]).tocsr()
S.eliminate_zeros()

# 6. Kiểm tra và lưu thành file mới
assert S.shape == (len(catalog), len(catalog))
assert S.shape[0] == len(catalog)

save_npz(
    os.path.join(ITEMCF_DIR, "item_similarity_shared.npz"),
    S
)
save_npz(
    os.path.join(ITEMCF_DIR, "user_item_binary_shared.npz"),
    X
)
np.save(
    os.path.join(ITEMCF_DIR, "item_catalog_shared.npy"),
    catalog
)
np.save(
    os.path.join(ITEMCF_DIR, "user_ids_shared.npy"),
    user_ids
)

print("\n===== ITEM-CF ĐÃ LƯU =====")
print("Similarity shape:", S.shape)
print("Similarity nnz:", S.nnz)
print("Kiểu dữ liệu similarity:", S.dtype)
print("Thư mục:", ITEMCF_DIR)

===== MA TRẬN KHÁCH HÀNG - SẢN PHẨM =====
Kích thước: (190980, 114800)
Số cặp user-item khác nhau: 590157

Đang tính ma trận đồng xuất hiện...
Kích thước co-occurrence: (114800, 114800)
Số phần tử khác 0: 63058346

Đang tính cosine similarity...

===== ITEM-CF ĐÃ LƯU =====
Similarity shape: (114800, 114800)
Similarity nnz: 63058346
Kiểu dữ liệu similarity: float32
Thư mục: /content/drive/MyDrive/BigData_ML3_ToNhu/itemcf_shared


In [9]:
import os
import json
import numpy as np
from scipy.sparse import load_npz, csr_matrix

# 1. Đường dẫn
BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
DATA_DIR = os.path.join(BASE_DIR, "data_checkpoint")
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")
ITEMCF_DIR = os.path.join(BASE_DIR, "itemcf_shared")
RESULTS_DIR = os.path.join(BASE_DIR, "results_shared")
os.makedirs(RESULTS_DIR, exist_ok=True)

# 2. Đọc ma trận Item-CF và dữ liệu đánh giá
X = load_npz(
    os.path.join(ITEMCF_DIR, "user_item_binary_shared.npz")
).tocsr()

S = load_npz(
    os.path.join(ITEMCF_DIR, "item_similarity_shared.npz")
).tocsr()

catalog = np.load(
    os.path.join(ITEMCF_DIR, "item_catalog_shared.npy")
)
user_ids = np.load(
    os.path.join(ITEMCF_DIR, "user_ids_shared.npy")
)
eval_users = np.load(
    os.path.join(EVAL_DIR, "eval_users_shared.npy")
)
test_candidates = np.load(
    os.path.join(EVAL_DIR, "test_candidates_shared.npy")
)
test_priorities = np.load(
    os.path.join(EVAL_DIR, "tie_priorities_test_shared.npy")
)

assert test_candidates.shape == (len(eval_users), 101)
assert test_priorities.shape == test_candidates.shape
assert X.shape == (len(user_ids), len(catalog))
assert S.shape == (len(catalog), len(catalog))

# 3. Tạo lịch sử test = train + target validation
train_df = __import__("pandas").read_parquet(
    os.path.join(DATA_DIR, "train_interactions_shared.parquet")
)
valid_df = __import__("pandas").read_parquet(
    os.path.join(DATA_DIR, "valid_targets_shared.parquet")
)

user_rows = np.searchsorted(user_ids, eval_users)
assert np.array_equal(user_ids[user_rows], eval_users)

H_train = X[user_rows].tocsr()

valid_map = dict(zip(
    valid_df["visitorid"].astype(int),
    valid_df["itemid"].astype(int)
))
valid_items = np.array(
    [valid_map[int(uid)] for uid in eval_users],
    dtype=np.int64
)

valid_cols = np.searchsorted(catalog, valid_items)
assert np.array_equal(catalog[valid_cols], valid_items)

V = csr_matrix(
    (
        np.ones(len(eval_users), dtype=np.int32),
        (np.arange(len(eval_users)), valid_cols)
    ),
    shape=H_train.shape,
    dtype=np.int32
)

H_test = (H_train + V).tocsr()
H_test.data[:] = 1
H_test.eliminate_zeros()
H_test.sort_indices()

# 4. Đổi các item ID trong candidate sang chỉ số cột
candidate_cols = np.searchsorted(catalog, test_candidates)
assert np.array_equal(catalog[candidate_cols], test_candidates)

# 5. Đánh giá theo batch, sử dụng đúng priority chung
ranks = np.empty(len(eval_users), dtype=np.int16)
batch_size = 256

for start in range(0, len(eval_users), batch_size):
    end = min(start + batch_size, len(eval_users))

    score_matrix = (H_test[start:end] @ S).tocsr()
    score_matrix.sort_indices()

    for j in range(end - start):
        row_start = score_matrix.indptr[j]
        row_end = score_matrix.indptr[j + 1]

        row_items = score_matrix.indices[row_start:row_end]
        row_scores = score_matrix.data[row_start:row_end]

        cand = candidate_cols[start + j]
        positions = np.searchsorted(row_items, cand)

        scores = np.zeros(len(cand), dtype=np.float32)
        in_range = positions < len(row_items)
        idx = np.flatnonzero(in_range)

        if len(idx):
            matched = row_items[positions[idx]] == cand[idx]
            matched_idx = idx[matched]
            scores[matched_idx] = row_scores[
                positions[matched_idx]
            ]

        priorities = test_priorities[start + j]
        positive_score = scores[0]
        positive_priority = priorities[0]

        ahead = (
            (scores[1:] > positive_score)
            | (
                (scores[1:] == positive_score)
                & (priorities[1:] < positive_priority)
            )
        )

        ranks[start + j] = 1 + int(ahead.sum())

    if end % 25000 < batch_size or end == len(eval_users):
        print(
            f"Đã đánh giá {end:,}/{len(eval_users):,} khách hàng"
        )

    del score_matrix

# 6. Tính chỉ số
hits = ranks <= 10
ndcg = np.where(
    hits,
    1.0 / np.log2(ranks + 1),
    0.0
)

metrics = {
    "model": "Item-CF",
    "split": "TEST",
    "tie_breaking": "shared_random_priorities",
    "num_users": int(len(ranks)),
    "num_candidates_per_user": 101,
    "HR@10": float(hits.mean()),
    "NDCG@10": float(ndcg.mean()),
    "rank_1": int(np.sum(ranks == 1)),
    "rank_le_10": int(np.sum(ranks <= 10)),
    "misses": int(np.sum(ranks > 10)),
    "median_rank": float(np.median(ranks)),
    "mean_rank": float(np.mean(ranks)),
    "max_rank": int(np.max(ranks))
}

# 7. Lưu kết quả hiệu chỉnh riêng
np.save(
    os.path.join(
        RESULTS_DIR,
        "itemcf_test_positive_rank_shared_fair_common.npy"
    ),
    ranks
)

with open(
    os.path.join(
        RESULTS_DIR,
        "itemcf_test_metrics_shared_fair_common.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\n===== ITEM-CF — TEST, PHÁ HÒA CHUNG =====")
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("\nKết quả đã lưu tại:", RESULTS_DIR)

Đã đánh giá 25,088/172,513 khách hàng
Đã đánh giá 50,176/172,513 khách hàng
Đã đánh giá 75,008/172,513 khách hàng
Đã đánh giá 100,096/172,513 khách hàng
Đã đánh giá 125,184/172,513 khách hàng
Đã đánh giá 150,016/172,513 khách hàng
Đã đánh giá 172,513/172,513 khách hàng

===== ITEM-CF — TEST, PHÁ HÒA CHUNG =====
{
  "model": "Item-CF",
  "split": "TEST",
  "tie_breaking": "shared_random_priorities",
  "num_users": 172513,
  "num_candidates_per_user": 101,
  "HR@10": 0.5980650733567905,
  "NDCG@10": 0.4976560175418854,
  "rank_1": 69557,
  "rank_le_10": 103174,
  "misses": 69339,
  "median_rank": 3.0,
  "mean_rank": 23.332003964918584,
  "max_rank": 101
}

Kết quả đã lưu tại: /content/drive/MyDrive/BigData_ML3_ToNhu/results_shared


## 6. Chuẩn bị dữ liệu tuần tự cho SASRec

Lập chỉ mục sản phẩm từ train catalog; index `0` dành cho padding. Chuỗi đầu vào có độ dài tối đa 50; validation dùng train history, test dùng train history cộng validation target.


In [10]:
import os
import json
import pickle
import numpy as np
import pandas as pd

# 1. Đường dẫn dữ liệu chung và thư mục SASRec mới
BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
DATA_DIR = os.path.join(BASE_DIR, "data_checkpoint")
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")
SASREC_SHARED_DIR = os.path.join(BASE_DIR, "sasrec_shared")

os.makedirs(SASREC_SHARED_DIR, exist_ok=True)

MAXLEN = 50

# 2. Đọc dữ liệu train và bộ ứng viên đã tạo từ nguồn chung
train_df = pd.read_parquet(
    os.path.join(DATA_DIR, "train_interactions_shared.parquet")
)

eval_users = np.load(
    os.path.join(EVAL_DIR, "eval_users_shared.npy")
)

valid_candidates_raw = np.load(
    os.path.join(EVAL_DIR, "valid_candidates_shared.npy")
)

test_candidates_raw = np.load(
    os.path.join(EVAL_DIR, "test_candidates_shared.npy")
)

assert valid_candidates_raw.shape == (len(eval_users), 101)
assert test_candidates_raw.shape == (len(eval_users), 101)

# 3. Đảm bảo thứ tự tương tác đúng theo thời gian
train_df = train_df.sort_values(
    ["visitorid", "timestamp"],
    kind="mergesort"
).reset_index(drop=True)

# 4. Tạo vocabulary item; index 0 dành cho padding
catalog = np.sort(
    train_df["itemid"].unique()
).astype(np.int64)

item_to_idx = {
    int(item_id): idx + 1
    for idx, item_id in enumerate(catalog)
}

NUM_ITEMS = len(catalog)

# 5. Chuyển chuỗi train sang index của SASRec
train_sequences_encoded = {}

for user_id, group in train_df.groupby(
    "visitorid", sort=False
):
    train_sequences_encoded[int(user_id)] = [
        item_to_idx[int(item)]
        for item in group["itemid"].to_numpy()
    ]

trainable_users = np.asarray(
    [
        user_id
        for user_id, seq in train_sequences_encoded.items()
        if len(seq) > 1
    ],
    dtype=np.int64
)

# 6. Chuyển ứng viên từ itemid gốc sang SASRec index
def map_candidate_matrix(candidates, name):
    positions = np.searchsorted(catalog, candidates)

    if np.any(positions >= len(catalog)):
        raise ValueError(
            f"{name}: có sản phẩm không thuộc train catalog."
        )

    if not np.array_equal(catalog[positions], candidates):
        raise ValueError(
            f"{name}: có sản phẩm không thuộc train catalog."
        )

    return (positions + 1).astype(np.int32)

valid_candidates = map_candidate_matrix(
    valid_candidates_raw, "Validation candidates"
)

test_candidates = map_candidate_matrix(
    test_candidates_raw, "Test candidates"
)

# 7. Tạo chuỗi validation và test, left-padding đến MAXLEN
def left_pad_sequence(sequence, maxlen=MAXLEN):
    result = np.zeros(maxlen, dtype=np.int32)
    recent = sequence[-maxlen:]

    if recent:
        result[-len(recent):] = recent

    return result

valid_seq_input = np.zeros(
    (len(eval_users), MAXLEN), dtype=np.int32
)
test_seq_input = np.zeros(
    (len(eval_users), MAXLEN), dtype=np.int32
)

for i, user_id in enumerate(eval_users):
    user_id = int(user_id)

    if user_id not in train_sequences_encoded:
        raise ValueError(
            f"Không tìm thấy train sequence của user {user_id}"
        )

    train_seq = train_sequences_encoded[user_id]

    # Validation: chỉ dùng lịch sử train
    valid_seq_input[i] = left_pad_sequence(train_seq)

    # Test: dùng train history cộng target validation
    valid_item_raw = int(valid_candidates_raw[i, 0])
    valid_item_idx = item_to_idx[valid_item_raw]

    test_seq_input[i] = left_pad_sequence(
        train_seq + [valid_item_idx]
    )

# 8. Kiểm tra cấu trúc dữ liệu
assert NUM_ITEMS == 114800
assert valid_seq_input.shape == (len(eval_users), MAXLEN)
assert test_seq_input.shape == (len(eval_users), MAXLEN)
assert valid_candidates.shape == (len(eval_users), 101)
assert test_candidates.shape == (len(eval_users), 101)
assert np.all(valid_candidates[:, 0] > 0)
assert np.all(test_candidates[:, 0] > 0)

# 9. Lưu file mới; không ghi đè SASRec hoặc kết quả cũ
files_to_save = {
    "train_sequences_encoded.pkl": train_sequences_encoded,
    "item_to_idx.pkl": item_to_idx,
}

for filename, obj in files_to_save.items():
    with open(
        os.path.join(SASREC_SHARED_DIR, filename), "wb"
    ) as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

np.save(
    os.path.join(SASREC_SHARED_DIR, "trainable_users.npy"),
    trainable_users
)
np.save(
    os.path.join(SASREC_SHARED_DIR, "eval_users.npy"),
    eval_users
)
np.save(
    os.path.join(SASREC_SHARED_DIR, "eval_valid_seq.npy"),
    valid_seq_input
)
np.save(
    os.path.join(SASREC_SHARED_DIR, "eval_test_seq.npy"),
    test_seq_input
)
np.save(
    os.path.join(SASREC_SHARED_DIR, "eval_valid_candidates.npy"),
    valid_candidates
)
np.save(
    os.path.join(SASREC_SHARED_DIR, "eval_test_candidates.npy"),
    test_candidates
)

metadata = {
    "source": "clean_events_final.csv",
    "maxlen": MAXLEN,
    "num_items": int(NUM_ITEMS),
    "num_train_users": int(len(train_sequences_encoded)),
    "num_trainable_users": int(len(trainable_users)),
    "num_eval_users": int(len(eval_users)),
    "candidate_size": 101,
    "positive_column": 0,
    "seeds": [2026, 2027, 2028]
}

with open(
    os.path.join(SASREC_SHARED_DIR, "metadata.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("===== SASRec SHARED DATA PREPARATION =====")
print("Số sản phẩm:", NUM_ITEMS)
print("Số user có train sequence:", len(train_sequences_encoded))
print("Số user tạo được training target:", len(trainable_users))
print("Validation sequence:", valid_seq_input.shape)
print("Test sequence:", test_seq_input.shape)
print("Validation candidates:", valid_candidates.shape)
print("Test candidates:", test_candidates.shape)
print("Padding index: 0")
print("Thư mục lưu:", SASREC_SHARED_DIR)
print("Đã kiểm tra các target ứng viên thuộc train catalog.")

===== SASRec SHARED DATA PREPARATION =====
Số sản phẩm: 114800
Số user có train sequence: 190980
Số user tạo được training target: 113451
Validation sequence: (172513, 50)
Test sequence: (172513, 50)
Validation candidates: (172513, 101)
Test candidates: (172513, 101)
Padding index: 0
Thư mục lưu: /content/drive/MyDrive/BigData_ML3_ToNhu/sasrec_shared
Đã kiểm tra các target ứng viên thuộc train catalog.


In [11]:

import os
import pickle
import numpy as np
import torch

BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
SASREC_DIR = os.path.join(BASE_DIR, "sasrec_shared")

required_files = [
    "train_sequences_encoded.pkl",
    "trainable_users.npy",
    "eval_valid_seq.npy",
    "eval_valid_candidates.npy",
    "eval_test_seq.npy",
    "eval_test_candidates.npy",
    "item_to_idx.pkl",
]

print("===== CHẨN ĐOÁN DỮ LIỆU SASRec =====")
print("Thư mục:", SASREC_DIR)
print("Thư mục tồn tại:", os.path.isdir(SASREC_DIR))
print("GPU khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n===== KIỂM TRA FILE =====")
missing_files = []

for name in required_files:
    path = os.path.join(SASREC_DIR, name)
    exists = os.path.isfile(path)
    print(f"{'✅' if exists else '❌'} {name}")

    if not exists:
        missing_files.append(name)

if missing_files:
    print("\n⚠️ Các file đang thiếu:")
    for name in missing_files:
        print("-", os.path.join(SASREC_DIR, name))

    print(
        "\nTạm dừng kiểm tra dữ liệu. "
        "Chưa huấn luyện và chưa xóa hoặc ghi đè file nào."
    )

else:
    try:
        with open(
            os.path.join(SASREC_DIR, "train_sequences_encoded.pkl"),
            "rb"
        ) as f:
            train_sequences = pickle.load(f)

        with open(
            os.path.join(SASREC_DIR, "item_to_idx.pkl"),
            "rb"
        ) as f:
            item_to_idx = pickle.load(f)

        trainable_users = np.load(
            os.path.join(SASREC_DIR, "trainable_users.npy")
        )
        valid_seq = np.load(
            os.path.join(SASREC_DIR, "eval_valid_seq.npy")
        )
        valid_candidates = np.load(
            os.path.join(SASREC_DIR, "eval_valid_candidates.npy")
        )
        test_seq = np.load(
            os.path.join(SASREC_DIR, "eval_test_seq.npy")
        )
        test_candidates = np.load(
            os.path.join(SASREC_DIR, "eval_test_candidates.npy")
        )

        checks = {
            "Số sản phẩm bằng 114800": len(item_to_idx) == 114800,
            "Số chuỗi train bằng 190980": len(train_sequences) == 190980,
            "Số user trainable bằng 113451": len(trainable_users) == 113451,
            "Validation sequence đúng shape": valid_seq.shape == (172513, 50),
            "Validation candidates đúng shape": valid_candidates.shape == (172513, 101),
            "Test sequence đúng shape": test_seq.shape == (172513, 50),
            "Test candidates đúng shape": test_candidates.shape == (172513, 101),
        }

        print("\n===== KÍCH THƯỚC THỰC TẾ =====")
        print("Số chuỗi train:", len(train_sequences))
        print("Số user trainable:", len(trainable_users))
        print("Số sản phẩm:", len(item_to_idx))
        print("Validation sequence:", valid_seq.shape)
        print("Validation candidates:", valid_candidates.shape)
        print("Test sequence:", test_seq.shape)
        print("Test candidates:", test_candidates.shape)

        print("\n===== KẾT QUẢ KIỂM TRA =====")
        for description, passed in checks.items():
            print(f"{'✅' if passed else '❌'} {description}")

        if all(checks.values()):
            print("\n✅ Các file và kích thước dữ liệu SASRec hợp lệ.")
        else:
            print("\n⚠️ Có điều kiện chưa đạt, cần xem lại trước khi huấn luyện.")

        if not torch.cuda.is_available():
            print(
                "⚠️ Runtime hiện chưa có GPU. "
                "Chưa chạy cell huấn luyện SASRec."
            )

    except Exception as e:
        print("\n❌ Lỗi khi đọc hoặc kiểm tra dữ liệu:")
        print(type(e).__name__, str(e))


===== CHẨN ĐOÁN DỮ LIỆU SASRec =====
Thư mục: /content/drive/MyDrive/BigData_ML3_ToNhu/sasrec_shared
Thư mục tồn tại: True
GPU khả dụng: True
GPU: NVIDIA L4

===== KIỂM TRA FILE =====
✅ train_sequences_encoded.pkl
✅ trainable_users.npy
✅ eval_valid_seq.npy
✅ eval_valid_candidates.npy
✅ eval_test_seq.npy
✅ eval_test_candidates.npy
✅ item_to_idx.pkl

===== KÍCH THƯỚC THỰC TẾ =====
Số chuỗi train: 190980
Số user trainable: 113451
Số sản phẩm: 114800
Validation sequence: (172513, 50)
Validation candidates: (172513, 101)
Test sequence: (172513, 50)
Test candidates: (172513, 101)

===== KẾT QUẢ KIỂM TRA =====
✅ Số sản phẩm bằng 114800
✅ Số chuỗi train bằng 190980
✅ Số user trainable bằng 113451
✅ Validation sequence đúng shape
✅ Validation candidates đúng shape
✅ Test sequence đúng shape
✅ Test candidates đúng shape

✅ Các file và kích thước dữ liệu SASRec hợp lệ.


## 7. Huấn luyện SASRec

Huấn luyện ba seed (`2026`, `2027`, `2028`), tối đa 10 epoch mỗi seed. Checkpoint tốt nhất được chọn theo validation NDCG@10. Cell có kiểm tra metrics/checkpoint để bỏ qua seed đã hoàn tất nếu chạy lại. Tất cả checkpoint nằm trong Google Drive, không đưa vào Git.


In [12]:
import os
import json
import time
import random
import pickle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================================================
# 1. ĐƯỜNG DẪN VÀ CẤU HÌNH
# =========================================================
BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
SASREC_DIR = os.path.join(BASE_DIR, "sasrec_shared")

RUNS_DIR = os.path.join(SASREC_DIR, "runs")
os.makedirs(RUNS_DIR, exist_ok=True)

assert torch.cuda.is_available(), "Không tìm thấy GPU."
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

with open(
    os.path.join(SASREC_DIR, "train_sequences_encoded.pkl"), "rb"
) as f:
    train_sequences_encoded = pickle.load(f)

trainable_users = np.load(
    os.path.join(SASREC_DIR, "trainable_users.npy")
).astype(np.int64)

valid_seq_input = np.load(
    os.path.join(SASREC_DIR, "eval_valid_seq.npy"),
    mmap_mode="r"
)
valid_candidates = np.load(
    os.path.join(SASREC_DIR, "eval_valid_candidates.npy"),
    mmap_mode="r"
)

NUM_ITEMS = 114800
MAXLEN = 50
HIDDEN_UNITS = 50
NUM_BLOCKS = 2
NUM_HEADS = 1
DROPOUT_RATE = 0.5
LEARNING_RATE = 0.001
BATCH_SIZE = 128
EPOCHS = 10
EVAL_BATCH_SIZE = 1024
SEEDS = [2026, 2027, 2028]

NUM_BATCHES_PER_EPOCH = len(train_sequences_encoded) // BATCH_SIZE

assert len(train_sequences_encoded) == 190980
assert len(trainable_users) == 113451
assert valid_seq_input.shape == (172513, 50)
assert valid_candidates.shape == (172513, 101)

print("Train sequences:", len(train_sequences_encoded))
print("Trainable users:", len(trainable_users))
print("Validation sequences:", valid_seq_input.shape)
print("Validation candidates:", valid_candidates.shape)
print("Batches/epoch:", NUM_BATCHES_PER_EPOCH)

# =========================================================
# 2. MÔ HÌNH SASRec
# =========================================================
class FaithfulSASRec(nn.Module):
    def __init__(
        self,
        num_items,
        hidden_units=50,
        maxlen=50,
        num_blocks=2,
        num_heads=1,
        dropout_rate=0.5
    ):
        super().__init__()

        assert hidden_units % num_heads == 0

        self.num_items = num_items
        self.hidden_units = hidden_units
        self.maxlen = maxlen
        self.num_blocks = num_blocks
        self.num_heads = num_heads
        self.head_dim = hidden_units // num_heads

        self.item_emb = nn.Embedding(
            num_items + 1, hidden_units, padding_idx=0
        )
        self.pos_emb = nn.Embedding(maxlen, hidden_units)
        self.input_dropout = nn.Dropout(dropout_rate)

        self.attn_norms = nn.ModuleList()
        self.q_layers = nn.ModuleList()
        self.k_layers = nn.ModuleList()
        self.v_layers = nn.ModuleList()
        self.attn_dropout = nn.Dropout(dropout_rate)

        self.ffn_norms = nn.ModuleList()
        self.ffn_linear1 = nn.ModuleList()
        self.ffn_linear2 = nn.ModuleList()
        self.ffn_dropout1 = nn.Dropout(dropout_rate)
        self.ffn_dropout2 = nn.Dropout(dropout_rate)

        for _ in range(num_blocks):
            self.attn_norms.append(
                nn.LayerNorm(hidden_units, eps=1e-8)
            )
            self.q_layers.append(nn.Linear(hidden_units, hidden_units))
            self.k_layers.append(nn.Linear(hidden_units, hidden_units))
            self.v_layers.append(nn.Linear(hidden_units, hidden_units))

            self.ffn_norms.append(
                nn.LayerNorm(hidden_units, eps=1e-8)
            )
            self.ffn_linear1.append(
                nn.Linear(hidden_units, hidden_units)
            )
            self.ffn_linear2.append(
                nn.Linear(hidden_units, hidden_units)
            )

        self.final_norm = nn.LayerNorm(hidden_units, eps=1e-8)

        self.register_buffer(
            "causal_mask",
            torch.triu(
                torch.ones(maxlen, maxlen, dtype=torch.bool),
                diagonal=1
            ),
            persistent=False
        )

        self._initialize_weights()

    def _initialize_weights(self):
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.xavier_uniform_(self.pos_emb.weight)

        with torch.no_grad():
            self.item_emb.weight[0].zero_()

        for layers in [
            self.q_layers, self.k_layers, self.v_layers,
            self.ffn_linear1, self.ffn_linear2
        ]:
            for layer in layers:
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

        for norm in list(self.attn_norms) + list(self.ffn_norms) + [self.final_norm]:
            nn.init.ones_(norm.weight)
            nn.init.zeros_(norm.bias)

    def _split_heads(self, x):
        B, T, H = x.shape
        x = x.view(B, T, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def _merge_heads(self, x):
        B, heads, T, D = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, T, heads * D)

    def forward(self, input_seq):
        B, T = input_seq.shape

        valid_mask = input_seq != 0
        valid_mask_float = valid_mask.unsqueeze(-1).float()

        x = self.item_emb(input_seq) * (self.hidden_units ** 0.5)

        positions = torch.arange(T, device=input_seq.device).unsqueeze(0)
        x = self.input_dropout(x + self.pos_emb(positions))
        x = x * valid_mask_float

        for block in range(self.num_blocks):
            q_input = self.attn_norms[block](x)

            q = self._split_heads(self.q_layers[block](q_input))
            k = self._split_heads(self.k_layers[block](x))
            v = self._split_heads(self.v_layers[block](x))

            scores = torch.matmul(q, k.transpose(-2, -1))
            scores = scores / (self.head_dim ** 0.5)

            key_padding = (~valid_mask).unsqueeze(1).unsqueeze(1)
            causal = self.causal_mask[:T, :T].unsqueeze(0).unsqueeze(0)

            scores = scores.masked_fill(key_padding, -2**32 + 1)
            scores = scores.masked_fill(causal, -2**32 + 1)

            weights = torch.softmax(scores, dim=-1)
            weights = self.attn_dropout(weights)

            attn_output = self._merge_heads(torch.matmul(weights, v))

            x = (attn_output + q_input) * valid_mask_float

            ffn_input = self.ffn_norms[block](x)
            ffn_output = F.relu(self.ffn_linear1[block](ffn_input))
            ffn_output = self.ffn_dropout1(ffn_output)
            ffn_output = self.ffn_linear2[block](ffn_output)
            ffn_output = self.ffn_dropout2(ffn_output)

            x = (ffn_output + ffn_input) * valid_mask_float

        return self.final_norm(x) * valid_mask_float

# =========================================================
# 3. TẠO BATCH HUẤN LUYỆN
# =========================================================
def sample_negative(itemnum, user_item_set, rng):
    while True:
        item = int(rng.integers(1, itemnum + 1))
        if item not in user_item_set:
            return item

def make_training_batch(
    user_list, batch_size, maxlen, itemnum, sequences, rng
):
    selected_users = rng.choice(
        user_list, size=batch_size, replace=True
    )

    seq_batch = np.zeros((batch_size, maxlen), dtype=np.int64)
    pos_batch = np.zeros((batch_size, maxlen), dtype=np.int64)
    neg_batch = np.zeros((batch_size, maxlen), dtype=np.int64)

    for row, user_id in enumerate(selected_users):
        user_seq = sequences[int(user_id)]
        next_item = user_seq[-1]
        idx = maxlen - 1
        user_item_set = set(user_seq)

        for item in reversed(user_seq[:-1]):
            seq_batch[row, idx] = item
            pos_batch[row, idx] = next_item
            neg_batch[row, idx] = sample_negative(
                itemnum, user_item_set, rng
            )

            next_item = item
            idx -= 1

            if idx < 0:
                break

    return seq_batch, pos_batch, neg_batch

# =========================================================
# 4. ĐÁNH GIÁ VALIDATION
# =========================================================
@torch.inference_mode()
def evaluate_validation(model):
    model.eval()
    all_ranks = []

    for start in range(0, len(valid_seq_input), EVAL_BATCH_SIZE):
        end = min(start + EVAL_BATCH_SIZE, len(valid_seq_input))

        seq_np = np.array(
            valid_seq_input[start:end], dtype=np.int64, copy=True
        )
        cand_np = np.array(
            valid_candidates[start:end], dtype=np.int64, copy=True
        )

        seq = torch.from_numpy(seq_np).to(device)
        cand = torch.from_numpy(cand_np).to(device)

        hidden = model(seq)
        last_hidden = hidden[:, -1, :]
        candidate_emb = model.item_emb(cand)

        scores = torch.sum(
            last_hidden.unsqueeze(1) * candidate_emb, dim=-1
        )
        ranks = 1 + torch.sum(
            scores[:, 1:] > scores[:, [0]], dim=1
        )
        all_ranks.append(ranks.cpu().numpy().astype(np.int32))

    ranks = np.concatenate(all_ranks)
    hits = ranks <= 10
    ndcg = np.where(hits, 1.0 / np.log2(ranks + 1), 0.0)

    return float(hits.mean()), float(ndcg.mean()), ranks

# =========================================================
# 5. HUẤN LUYỆN TỪNG SEED
# =========================================================
def train_one_seed(seed):
    run_dir = os.path.join(RUNS_DIR, f"seed{seed}")
    os.makedirs(run_dir, exist_ok=True)

    best_path = os.path.join(run_dir, "best_checkpoint.pt")
    latest_path = os.path.join(run_dir, "latest_checkpoint.pt")
    metrics_path = os.path.join(run_dir, "metrics.json")

    # Nếu seed này đã hoàn thành đủ 10 epoch và có checkpoint,
    # không chạy lại khi em vô tình chạy lại cell.
    if os.path.exists(best_path) and os.path.exists(metrics_path):
        with open(metrics_path, "r", encoding="utf-8") as f:
            previous = json.load(f)
        if len(previous.get("history", [])) >= EPOCHS:
            print(f"\nSeed {seed} đã hoàn thành, bỏ qua.")
            return previous

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    rng = np.random.default_rng(seed)

    model = FaithfulSASRec(
        num_items=NUM_ITEMS,
        hidden_units=HIDDEN_UNITS,
        maxlen=MAXLEN,
        num_blocks=NUM_BLOCKS,
        num_heads=NUM_HEADS,
        dropout_rate=DROPOUT_RATE
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=(0.9, 0.98),
        weight_decay=0.0
    )

    best_ndcg = -1.0
    best_epoch = None
    history = []
    start_time = time.time()

    print("\n" + "=" * 65)
    print(f"BẮT ĐẦU HUẤN LUYỆN SASRec — SEED {seed}")
    print("=" * 65)

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()

        running_loss = 0.0
        running_positions = 0

        for step in range(NUM_BATCHES_PER_EPOCH):
            seq_np, pos_np, neg_np = make_training_batch(
                trainable_users, BATCH_SIZE, MAXLEN,
                NUM_ITEMS, train_sequences_encoded, rng
            )

            seq = torch.from_numpy(seq_np).to(device)
            pos = torch.from_numpy(pos_np).to(device)
            neg = torch.from_numpy(neg_np).to(device)

            optimizer.zero_grad(set_to_none=True)

            hidden = model(seq)
            pos_emb = model.item_emb(pos)
            neg_emb = model.item_emb(neg)

            pos_logits = torch.sum(hidden * pos_emb, dim=-1)
            neg_logits = torch.sum(hidden * neg_emb, dim=-1)
            mask = pos != 0

            if not mask.any():
                continue

            pos_loss = -F.logsigmoid(pos_logits)[mask].mean()
            neg_loss = -F.logsigmoid(-neg_logits)[mask].mean()
            loss = pos_loss + neg_loss

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Loss không hữu hạn: seed={seed}, epoch={epoch}, step={step}"
                )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=5.0
            )
            optimizer.step()

            n_positions = int(mask.sum().item())
            running_loss += float(loss.detach()) * n_positions
            running_positions += n_positions

            if (step + 1) % 250 == 0:
                print(
                    f"Seed {seed} | Epoch {epoch}/{EPOCHS} | "
                    f"Batch {step + 1}/{NUM_BATCHES_PER_EPOCH}"
                )

        avg_loss = running_loss / max(running_positions, 1)

        valid_hr, valid_ndcg, valid_ranks = evaluate_validation(model)
        epoch_time = time.time() - epoch_start

        row = {
            "epoch": epoch,
            "train_loss": float(avg_loss),
            "valid_hr10": valid_hr,
            "valid_ndcg10": valid_ndcg,
            "valid_median_rank": float(np.median(valid_ranks)),
            "epoch_time_sec": float(epoch_time)
        }
        history.append(row)

        print(
            f"\nSeed {seed} | Epoch {epoch}/{EPOCHS} | "
            f"Loss={avg_loss:.6f} | "
            f"VALID HR@10={valid_hr:.6f} | "
            f"VALID NDCG@10={valid_ndcg:.6f} | "
            f"Time={epoch_time / 60:.2f} phút"
        )

        checkpoint_data = {
            "epoch": epoch,
            "seed": seed,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "valid_hr10": valid_hr,
            "valid_ndcg10": valid_ndcg,
            "config": {
                "num_items": NUM_ITEMS,
                "maxlen": MAXLEN,
                "hidden_units": HIDDEN_UNITS,
                "num_blocks": NUM_BLOCKS,
                "num_heads": NUM_HEADS,
                "dropout_rate": DROPOUT_RATE,
                "learning_rate": LEARNING_RATE,
                "batch_size": BATCH_SIZE,
                "batches_per_epoch": NUM_BATCHES_PER_EPOCH
            }
        }

        # Lưu checkpoint gần nhất sau mỗi epoch
        torch.save(checkpoint_data, latest_path)

        if valid_ndcg > best_ndcg:
            best_ndcg = valid_ndcg
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "seed": seed,
                    "model_state_dict": model.state_dict(),
                    "valid_hr10": valid_hr,
                    "valid_ndcg10": valid_ndcg,
                    "config": checkpoint_data["config"]
                },
                best_path
            )
            print("  -> Đã lưu checkpoint tốt nhất.")

        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "seed": seed,
                    "best_epoch": best_epoch,
                    "best_valid_ndcg10": best_ndcg,
                    "history": history
                },
                f, ensure_ascii=False, indent=2
            )

        del valid_ranks, seq, pos, neg, hidden
        torch.cuda.empty_cache()

    print(
        f"\nHOÀN TẤT SEED {seed}. "
        f"Best epoch={best_epoch}, "
        f"best VALID NDCG@10={best_ndcg:.6f}"
    )
    print("Đã lưu tại:", run_dir)

    del model, optimizer
    torch.cuda.empty_cache()

    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_valid_ndcg10": best_ndcg,
        "history": history,
        "total_time_sec": time.time() - start_time
    }

# =========================================================
# 6. CHẠY 3 SEED
# =========================================================
seed_summaries = []

for seed in SEEDS:
    result = train_one_seed(seed)
    seed_summaries.append({
        "seed": seed,
        "best_epoch": result.get("best_epoch"),
        "best_valid_ndcg10": result.get("best_valid_ndcg10")
    })

summary_df = __import__("pandas").DataFrame(seed_summaries)
summary_path = os.path.join(SASREC_DIR, "shared_training_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n===== TÓM TẮT HUẤN LUYỆN SASRec =====")
print(summary_df.to_string(index=False))
print("\nBảng tổng hợp:", summary_path)
print("Checkpoint và metrics được lưu riêng theo từng seed.")

GPU: NVIDIA L4
Train sequences: 190980
Trainable users: 113451
Validation sequences: (172513, 50)
Validation candidates: (172513, 101)
Batches/epoch: 1492

Seed 2026 đã hoàn thành, bỏ qua.

Seed 2027 đã hoàn thành, bỏ qua.

Seed 2028 đã hoàn thành, bỏ qua.

===== TÓM TẮT HUẤN LUYỆN SASRec =====
 seed  best_epoch  best_valid_ndcg10
 2026          10           0.733483
 2027          10           0.729659
 2028          10           0.733808

Bảng tổng hợp: /content/drive/MyDrive/BigData_ML3_ToNhu/sasrec_shared/shared_training_summary.csv
Checkpoint và metrics được lưu riêng theo từng seed.


## 8. Đánh giá SASRec trên test

Đánh giá best checkpoint của từng seed trên cùng 172.513 user, 101 candidates/user và cùng test tie priorities; lưu chỉ số từng seed và trung bình/độ lệch chuẩn mẫu từ ba seed.


In [13]:
import os
import json
import numpy as np
import pandas as pd
import torch

BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
SASREC_DIR = os.path.join(BASE_DIR, "sasrec_shared")
RUNS_DIR = os.path.join(SASREC_DIR, "runs")
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")
RESULTS_DIR = os.path.join(BASE_DIR, "results_shared")

os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Đọc dữ liệu test và quy tắc phá hòa dùng chung
test_seq = np.load(
    os.path.join(SASREC_DIR, "eval_test_seq.npy"),
    mmap_mode="r"
)
test_candidates = np.load(
    os.path.join(SASREC_DIR, "eval_test_candidates.npy"),
    mmap_mode="r"
)
test_priorities = np.load(
    os.path.join(EVAL_DIR, "tie_priorities_test_shared.npy"),
    mmap_mode="r"
)

assert test_seq.shape == (172513, 50)
assert test_candidates.shape == (172513, 101)
assert test_priorities.shape == test_candidates.shape

# Lớp FaithfulSASRec đã được định nghĩa ở cell huấn luyện
assert "FaithfulSASRec" in globals(), (
    "Không tìm thấy lớp FaithfulSASRec trong phiên Colab. "
    "Hãy chạy lại cell định nghĩa mô hình SASRec trước."
)

@torch.inference_mode()
def evaluate_sasrec_with_shared_ties(model):
    model.eval()
    all_ranks = []
    batch_size = 1024

    for start in range(0, len(test_seq), batch_size):
        end = min(start + batch_size, len(test_seq))

        seq_np = np.array(test_seq[start:end], dtype=np.int64, copy=True)
        cand_np = np.array(
            test_candidates[start:end], dtype=np.int64, copy=True
        )
        priority_np = np.array(
            test_priorities[start:end], dtype=np.int16, copy=True
        )

        seq = torch.from_numpy(seq_np).to(device)
        cand = torch.from_numpy(cand_np).to(device)

        hidden = model(seq)
        last_hidden = hidden[:, -1, :]
        candidate_emb = model.item_emb(cand)

        scores = torch.sum(
            last_hidden.unsqueeze(1) * candidate_emb, dim=-1
        ).cpu().numpy()

        # Đánh giá đồng điểm theo cùng bộ priorities đã dùng cho baseline
        positive_score = scores[:, [0]]
        positive_priority = priority_np[:, [0]]

        ahead = (
            (scores[:, 1:] > positive_score)
            | (
                (scores[:, 1:] == positive_score)
                & (priority_np[:, 1:] < positive_priority)
            )
        )

        ranks = 1 + ahead.sum(axis=1)
        all_ranks.append(ranks.astype(np.int32))

    ranks = np.concatenate(all_ranks)
    hits = ranks <= 10
    ndcg = np.where(hits, 1.0 / np.log2(ranks + 1), 0.0)

    metrics = {
        "model": "SASRec",
        "split": "TEST",
        "tie_breaking": "shared_random_priorities",
        "num_users": int(len(ranks)),
        "num_candidates_per_user": 101,
        "HR@10": float(hits.mean()),
        "NDCG@10": float(ndcg.mean()),
        "rank_1": int(np.sum(ranks == 1)),
        "rank_le_10": int(np.sum(ranks <= 10)),
        "misses": int(np.sum(ranks > 10)),
        "median_rank": float(np.median(ranks)),
        "mean_rank": float(np.mean(ranks)),
        "max_rank": int(np.max(ranks))
    }

    return metrics, ranks

# Đánh giá checkpoint tốt nhất của mỗi seed
seed_metrics = []

for seed in [2026, 2027, 2028]:
    print(f"\n===== SASRec — SEED {seed} =====")

    checkpoint_path = os.path.join(
        RUNS_DIR, f"seed{seed}", "best_checkpoint.pt"
    )
    assert os.path.isfile(checkpoint_path), (
        f"Không tìm thấy checkpoint: {checkpoint_path}"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )

    model = FaithfulSASRec(
        num_items=114800,
        hidden_units=50,
        maxlen=50,
        num_blocks=2,
        num_heads=1,
        dropout_rate=0.5
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])

    metrics, ranks = evaluate_sasrec_with_shared_ties(model)
    metrics["seed"] = seed
    metrics["best_epoch"] = int(checkpoint["epoch"])

    np.save(
        os.path.join(
            RESULTS_DIR,
            f"sasrec_seed{seed}_test_positive_rank_shared_fair_common.npy"
        ),
        ranks
    )

    with open(
        os.path.join(
            RESULTS_DIR,
            f"sasrec_seed{seed}_test_metrics_shared_fair_common.json"
        ),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    seed_metrics.append(metrics)
    print(json.dumps(metrics, ensure_ascii=False, indent=2))

    del model, checkpoint, ranks
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Tổng hợp trung bình và độ lệch chuẩn mẫu
seed_df = pd.DataFrame(seed_metrics)

summary = {
    "model": "SASRec",
    "split": "TEST",
    "tie_breaking": "shared_random_priorities",
    "num_seeds": 3,
    "seeds": [2026, 2027, 2028],
    "num_users": 172513,
    "num_candidates_per_user": 101,
    "mean_HR@10": float(seed_df["HR@10"].mean()),
    "std_HR@10_sample": float(seed_df["HR@10"].std(ddof=1)),
    "mean_NDCG@10": float(seed_df["NDCG@10"].mean()),
    "std_NDCG@10_sample": float(seed_df["NDCG@10"].std(ddof=1))
}

seed_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "sasrec_test_metrics_by_seed_shared_fair_common.csv"
    ),
    index=False
)

with open(
    os.path.join(
        RESULTS_DIR,
        "sasrec_test_summary_shared_fair_common.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n===== SASRec — TEST, PHÁ HÒA CHUNG =====")
print(seed_df[["seed", "best_epoch", "HR@10", "NDCG@10"]].to_string(index=False))

print("\n===== TỔNG HỢP BA SEED =====")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nKết quả đã lưu tại:", RESULTS_DIR)


===== SASRec — SEED 2026 =====
{
  "model": "SASRec",
  "split": "TEST",
  "tie_breaking": "shared_random_priorities",
  "num_users": 172513,
  "num_candidates_per_user": 101,
  "HR@10": 0.8348588222336867,
  "NDCG@10": 0.7452744825928217,
  "rank_1": 113713,
  "rank_le_10": 144024,
  "misses": 28489,
  "median_rank": 1.0,
  "mean_rank": 8.120019940526221,
  "max_rank": 101,
  "seed": 2026,
  "best_epoch": 10
}

===== SASRec — SEED 2027 =====
{
  "model": "SASRec",
  "split": "TEST",
  "tie_breaking": "shared_random_priorities",
  "num_users": 172513,
  "num_candidates_per_user": 101,
  "HR@10": 0.8304127804861083,
  "NDCG@10": 0.74286476886914,
  "rank_1": 113627,
  "rank_le_10": 143257,
  "misses": 29256,
  "median_rank": 1.0,
  "mean_rank": 8.327841959736368,
  "max_rank": 101,
  "seed": 2027,
  "best_epoch": 10
}

===== SASRec — SEED 2028 =====
{
  "model": "SASRec",
  "split": "TEST",
  "tie_breaking": "shared_random_priorities",
  "num_users": 172513,
  "num_candidates_per_user"

## 9. Bảng so sánh cuối

Cell cuối đọc kết quả đã lưu của ba mô hình, xác nhận chúng dùng cùng benchmark/tie policy, rồi lưu CSV và JSON. Trong JSON, độ lệch chuẩn chưa được định nghĩa cho baseline một seed được ghi là `null`.


In [14]:
import os
import json
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/BigData_ML3_ToNhu"
RESULTS_DIR = os.path.join(BASE_DIR, "results_shared")
os.makedirs(RESULTS_DIR, exist_ok=True)

result_files = {
    "Popularity": "popularity_test_metrics_shared_fair.json",
    "Item-CF": "itemcf_test_metrics_shared_fair_common.json",
    "SASRec": "sasrec_test_summary_shared_fair_common.json",
}
loaded = {}
for model_name, filename in result_files.items():
    path = os.path.join(RESULTS_DIR, filename)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Thiếu kết quả {model_name}: {path}")
    with open(path, "r", encoding="utf-8") as f:
        loaded[model_name] = json.load(f)

popularity = loaded["Popularity"]
itemcf = loaded["Item-CF"]
sasrec = loaded["SASRec"]

for model_name, result in loaded.items():
    assert result["num_users"] == 172513, f"{model_name}: số user không khớp."
    assert result["num_candidates_per_user"] == 101, f"{model_name}: số ứng viên không khớp."
    assert result["tie_breaking"] == "shared_random_priorities", f"{model_name}: tie policy không khớp."
assert sasrec["num_seeds"] == 3
assert sasrec["seeds"] == [2026, 2027, 2028]

comparison = pd.DataFrame([
    {
        "Model": "Popularity", "Test users": popularity["num_users"],
        "Candidates/user": popularity["num_candidates_per_user"],
        "HR@10": popularity["HR@10"], "HR@10 std": None,
        "NDCG@10": popularity["NDCG@10"], "NDCG@10 std": None,
    },
    {
        "Model": "Item-CF", "Test users": itemcf["num_users"],
        "Candidates/user": itemcf["num_candidates_per_user"],
        "HR@10": itemcf["HR@10"], "HR@10 std": None,
        "NDCG@10": itemcf["NDCG@10"], "NDCG@10 std": None,
    },
    {
        "Model": "SASRec (mean 3 seeds)", "Test users": sasrec["num_users"],
        "Candidates/user": sasrec["num_candidates_per_user"],
        "HR@10": sasrec["mean_HR@10"], "HR@10 std": sasrec["std_HR@10_sample"],
        "NDCG@10": sasrec["mean_NDCG@10"], "NDCG@10 std": sasrec["std_NDCG@10_sample"],
    },
])
comparison["HR@10 rank"] = comparison["HR@10"].rank(ascending=False, method="min").astype(int)
comparison["NDCG@10 rank"] = comparison["NDCG@10"].rank(ascending=False, method="min").astype(int)

csv_path = os.path.join(RESULTS_DIR, "ml3_comparison_test_shared_fair_common.csv")
json_path = os.path.join(RESULTS_DIR, "ml3_comparison_test_shared_fair_common.json")
comparison.to_csv(csv_path, index=False, encoding="utf-8-sig")

comparison_clean = comparison.astype(object).where(pd.notna(comparison), None)
records = comparison_clean.to_dict(orient="records")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2, allow_nan=False)

with open(json_path, "r", encoding="utf-8") as f:
    verified = json.load(f)
assert len(verified) == 3
assert [r["Model"] for r in verified] == ["Popularity", "Item-CF", "SASRec (mean 3 seeds)"]
assert all(verified[i][k] is None for i in [0, 1] for k in ["HR@10 std", "NDCG@10 std"])

print("===== BẢNG SO SÁNH ML3 — TEST, PHÁ HÒA CHUNG =====")
presentation = comparison.copy()
presentation["HR@10 std"] = presentation["HR@10 std"].map(lambda x: "—" if pd.isna(x) else f"{x:.6f}")
presentation["NDCG@10 std"] = presentation["NDCG@10 std"].map(lambda x: "—" if pd.isna(x) else f"{x:.6f}")
display(presentation.round(6))
print("\n✅ Ba mô hình: 172.513 user, 101 candidates/user, shared tie priorities.")
print("✅ SASRec mean/std được tổng hợp từ 3 seed: 2026, 2027, 2028.")
print("✅ JSON hợp lệ: độ lệch chuẩn không áp dụng được ghi thành null.")
print("CSV:", csv_path)
print("JSON:", json_path)


===== BẢNG SO SÁNH ML3 — TEST, PHÁ HÒA CHUNG =====


,Model,Test users,Candidates/user,HR@10,HR@10 std,NDCG@10,NDCG@10 std,HR@10 rank,NDCG@10 rank
0,Popularity,172513,101,0.535247,—,0.318512,—,3,3
1,Item-CF,172513,101,0.598065,—,0.497656,—,2,2
2,SASRec (mean 3 seeds),172513,101,0.833929,0.003156,0.744778,0.001719,1,1



✅ Ba mô hình: 172.513 user, 101 candidates/user, shared tie priorities.
✅ SASRec mean/std được tổng hợp từ 3 seed: 2026, 2027, 2028.
✅ JSON hợp lệ: độ lệch chuẩn không áp dụng được ghi thành null.
CSV: /content/drive/MyDrive/BigData_ML3_ToNhu/results_shared/ml3_comparison_test_shared_fair_common.csv
JSON: /content/drive/MyDrive/BigData_ML3_ToNhu/results_shared/ml3_comparison_test_shared_fair_common.json


## Ghi chú diễn giải kết quả

Benchmark này là **next-view target ranking trên tập ứng viên lấy mẫu cố định**, không phải đánh giá toàn bộ catalog và không đo purchase/conversion. Popularity dùng số lượt view trong train; Item-CF dùng tương tác user–item nhị phân; SASRec dùng chuỗi hành vi tuần tự. Tất cả mô hình được chấm trên cùng tập user/candidate và cùng tie priorities.

Ba lần SASRec đều chọn checkpoint tại epoch 10, là giới hạn epoch trong cấu hình hiện tại. Kết quả thể hiện hiệu năng tương đối ổn định giữa ba seed, nhưng chưa đủ để kết luận mô hình đã hội tụ.


In [15]:
import torch

print("GPU khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU khả dụng: True
GPU: NVIDIA L4
